# Блокнот 14. От RNN к трансформеру

Курс «Машинное обучение с нуля», модуль 14. Разделы: 1 — урок 14.2, 2 — урок 14.3, 3 — урок 14.4, 4 — урок 14.5. Выполняйте ячейки сверху вниз; там, где `...  # ваш код`, допишите решение.

Всё работает на процессоре за несколько минут; с видеокартой в Colab — быстрее.

In [ ]:
import math
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device)

## 1. RNN и LSTM: «назови первое слово» (урок 14.2)

Сеть читает последовательность из T случайных цифр и должна назвать первую. Простая RNN при T = 30 не справляется, LSTM со смещением вентиля забывания 3 — справляется.

**Задание 1.** В LSTM PyTorch смещения вентилей лежат подряд: входной, забывания, кандидат, выходной — по `hidden` чисел. Запишите в часть вентиля забывания значение 1,5 (смещений два вектора — `bias_ih` и `bias_hh`, вместе получится 3).

In [ ]:
class FirstToken(nn.Module):
    def __init__(self, cell, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(10, 16)
        self.rnn = {"RNN": nn.RNN, "LSTM": nn.LSTM}[cell](16, hidden, batch_first=True)
        if cell == "LSTM":
            for name, p in self.rnn.named_parameters():
                if "bias" in name:
                    ...  # ваш код: срез вентиля забывания
        self.head = nn.Linear(hidden, 10)

    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.head(out[:, -1])


def train_first(cell, T, steps):
    torch.manual_seed(0)
    net = FirstToken(cell).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    for _ in range(steps):
        x = torch.randint(0, 10, (64, T), device=device)
        opt.zero_grad()
        F.cross_entropy(net(x), x[:, 0]).backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
    with torch.no_grad():
        x = torch.randint(0, 10, (1000, T), device=device)
        return (net(x).argmax(1) == x[:, 0]).float().mean().item()


for cell, steps in (("RNN", 1500), ("LSTM", 500)):
    t0 = time.time()
    acc = train_first(cell, 30, steps)
    print(f"{cell}, T = 30, {steps} шагов: точность {acc:.3f} ({time.time() - t0:.0f} с)")
assert train_first("LSTM", 30, 500) > 0.95

## 2. Внимание своими руками (урок 14.3)

**Задание 2.** Допишите масштабированное скалярное внимание: сходства $QK^T/\sqrt{d_k}$, softmax по строкам, смесь значений. Необязательный аргумент `mask` — булева матрица, где `False` означает «смотреть нельзя».

In [ ]:
def attention(Q, K, V, mask=None):
    ...  # ваш код: верните (выход, веса)


X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
Wv = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
out, A = attention(X, X, X @ Wv)
print("веса:\n", A.round(decimals=4), "\nвыход:\n", out.round(decimals=4))
assert torch.allclose(out[0], torch.tensor([2.5989, 4.0]), atol=1e-3)
assert torch.allclose(out, F.scaled_dot_product_attention(X, X, X @ Wv), atol=1e-5)

causal = torch.tril(torch.ones(3, 3, dtype=torch.bool))
out_c, A_c = attention(X, X, X @ Wv, mask=causal)
print("с причинной маской:\n", A_c.round(decimals=4))
assert torch.allclose(A_c[0], torch.tensor([1.0, 0.0, 0.0]))
assert torch.allclose(out_c, F.scaled_dot_product_attention(X, X, X @ Wv, is_causal=True), atol=1e-5)

## 3. Блок трансформера (урок 14.4)

Сверим число параметров готового блока PyTorch с формулой урока: внимание $4d^2 + 4d$, FFN $d \to 4d \to d$, две LayerNorm.

In [ ]:
d = 512
layer = nn.TransformerEncoderLayer(d_model=d, nhead=8, dim_feedforward=4 * d)
by_formula = (4 * d * d + 4 * d) + (d * 4 * d + 4 * d + 4 * d * d + d) + 2 * 2 * d
print("PyTorch:", sum(p.numel() for p in layer.parameters()), "| по формуле:", by_formula)
assert sum(p.numel() for p in layer.parameters()) == by_formula == 3_152_384

**Задание 3.** Допишите `forward` своего блока в варианте pre-LN: `x = x + внимание(LN1(x))`, затем `x = x + FFN(LN2(x))`. Многоголовое внимание — `nn.MultiheadAttention`, ему передаётся причинная маска.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        n = x.shape[1]
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=x.device), 1)  # True — запрещено (так у MultiheadAttention)
        ...  # ваш код: две остаточные связи
        return x


blk = Block(64, 4)
y = blk(torch.randn(2, 10, 64))
assert y.shape == (2, 10, 64)
# причинность: изменение последнего токена не должно менять выходы предыдущих
x1 = torch.randn(1, 10, 64)
x2 = x1.clone()
x2[0, -1] += 1.0
assert torch.allclose(blk(x1)[0, :-1], blk(x2)[0, :-1], atol=1e-5)
print("блок работает и не подсматривает в будущее")

## 4. Маленькая GPT на символах (урок 14.5)

Обучим крошечную GPT предсказывать следующий **символ** небольшого текста. Токены — отдельные буквы, контекст — 64 символа. Через несколько сотен шагов модель начнёт порождать «похожий» текст.

In [ ]:
text = """Дроид чинит корабль. Корабль летит к звёздам. Звёзды светят над пустыней.
Пилот проверяет двигатель, дроид проверяет крылья, а штурман прокладывает курс.
Корабль взлетает, корабль садится, корабль снова взлетает.
Дроид говорит: курс проложен. Пилот говорит: двигатель готов. Штурман говорит: летим.
Над пустыней два солнца. Над лесом две луны. Над морем одна планета.
Маленький дроид знает дорогу, большой корабль знает небо, а пилот знает дроида.
""" * 20
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
V, CTX = len(chars), 64
print("символов в словаре:", V, "| длина текста:", len(text))


class TinyGPT(nn.Module):
    def __init__(self, V, d=96, heads=4, layers=2, ctx=CTX):
        super().__init__()
        self.tok = nn.Embedding(V, d)
        self.pos = nn.Embedding(ctx, d)            # выученные позиции, как в GPT-2
        self.blocks = nn.Sequential(*[Block(d, heads) for _ in range(layers)])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, V)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        return self.head(self.ln(self.blocks(x)))


gpt = TinyGPT(V).to(device)
print("параметров:", sum(p.numel() for p in gpt.parameters()))
opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
STEPS = 600
for step in range(STEPS + 1):
    ix = torch.randint(0, len(data) - CTX - 1, (32,), device=device)
    x = torch.stack([data[i:i + CTX] for i in ix])
    y = torch.stack([data[i + 1:i + CTX + 1] for i in ix])     # цель — тот же текст, сдвинутый на 1
    loss = F.cross_entropy(gpt(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 150 == 0:
        print(f"шаг {step}: потеря {loss.item():.3f}  (наугад было бы ln {V} = {math.log(V):.2f})")
assert loss.item() < 1.0

**Задание 4.** Допишите авторегрессивное порождение: на каждом шаге взять логиты последней позиции, превратить в вероятности (softmax с температурой) и выбрать следующий символ `torch.multinomial`, затем дописать его к контексту.

In [ ]:
@torch.no_grad()
def generate(start, n, temperature=0.8):
    gpt.eval()
    idx = torch.tensor([[stoi[c] for c in start]], device=device)
    for _ in range(n):
        logits = gpt(idx[:, -CTX:])[:, -1] / temperature
        ...  # ваш код: вероятности, выбор, дописать к idx
    return "".join(chars[i] for i in idx[0].tolist())


torch.manual_seed(1)
sample = generate("Дроид ", 200)
print(sample)
assert len(sample) == 206

Модель почти дословно воспроизводит обучающий текст: шесть строк, повторённых 20 раз, она просто **запомнила** — потеря упала почти до нуля. Это переобучение в чистом виде: чтобы модель обобщала, нужны гораздо больше данных и разнообразия (настоящие LLM учатся на триллионах токенов, модуль 15).

Поиграйте с температурой: при 0,2 модель повторяет самые частые фразы, при 1,5 — порождает почти случайные буквы. Подробно о температуре — в уроке 15.1.